In [2]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import folium

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
BOUNDARIES_DIR = DATA_DIR / "boundaries"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

tocs_path       = BOUNDARIES_DIR / "tocs" / "toc_jobs_full_2022.gpkg"
villages_path   = BOUNDARIES_DIR / "villages" / "village_jobs_full_2022.gpkg"

tocs_path, villages_path

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/tocs/toc_jobs_full_2022.gpkg'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/villages/village_jobs_full_2022.gpkg'))

In [3]:
toc = gpd.read_file(tocs_path)
villages = gpd.read_file(villages_path)

In [4]:
toc.columns, villages.columns

(Index(['OBJECTID', 'APPLICABIL', 'TOC_ID', 'income_weighted', 'rent_weighted',
        'hh_total', 'median_income_approx', 'median_rent_approx',
        'intersect_acres', 'ASNQE001', 'ASS8E001', 'ASS8E002', 'ASS8E003',
        'ASS9E002', 'ASS9E003', 'ASORE001', 'ASORE003', 'ASORE004', 'ASORE010',
        'ASORE018', 'ASORE019', 'ASORE021', 'ASORE002', 'pct_drive_alone',
        'pct_transit', 'pct_bike', 'pct_walk', 'pct_wfh', 'pct_auto',
        'jobs_total', 'jobs_per_housing_unit', 'jobs_per_household',
        'geometry'],
       dtype='object'),
 Index(['OBJECTID', 'NAME', 'intersect_acres', 'hh_weighted', 'income_weighted',
        'rent_weighted', 'ASNQE001_w', 'ASS8E001_w', 'ASS8E002_w', 'ASS8E003_w',
        'ASS9E002_w', 'ASS9E003_w', 'ASORE001_w', 'ASORE003_w', 'ASORE004_w',
        'ASORE010_w', 'ASORE018_w', 'ASORE019_w', 'ASORE021_w', 'ASORE002_w',
        'median_income_approx', 'median_rent_approx', 'ASNQE001', 'ASS8E001',
        'ASS8E002', 'ASS8E003', 'ASS9E002', 

In [6]:
# Reproject to WGS84 for web maps
toc_web     = toc.to_crs(4326).copy()
villages_web = villages.to_crs(4326).copy()

In [7]:
# Map time!

# --- Base: villages in grey for context ---

m_jobs = villages_web.explore(
    style_kwds={
        "fillOpacity": 0.2,
        "weight": 0.3,
        "color": "grey",
    },
    tooltip=[
        "NAME",
        "jobs_total",
        "jobs_per_housing_unit",
        "jobs_per_household"
    ],
    name="Villages – Jobs to Housing Ratio",
)

# --- TOC choropleth by transit mode share ---
toc_web.explore(
    m=m_jobs,
    column="jobs_per_housing_unit",
    cmap="Blues",
    legend=True,
    legend_kwds={"caption": "Jobs per Housing Unit (TOC)"},
    style_kwds={
        "fillOpacity": 0.7,
        "weight": 0.7,
        "color": "black",
    },
    tooltip=[
        "APPLICABIL",
        "jobs_total",
        "jobs_per_housing_unit",
        "jobs_per_household"
    ],
    name="TOCs – Jobs to Housing Ratio",
)

folium.LayerControl(collapsed=False).add_to(m_jobs)

jobs_map_path = OUTPUTS_DIR / "tocs_jobs_per_housing_unit.html"
m_jobs.save(jobs_map_path)

print("Saved:", jobs_map_path)

Saved: c:\Users\arjav\DevSource\toc-performance-dashboard\outputs\tocs_jobs_per_housing_unit.html
